# 04 · Metrik & Walk-Forward — Bab 5

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 5: mendiagnosa overfit (learning curve), menerapkan regularisasi, menghitung metrik regresi (MAE/RMSE/R²/Willmott/KGE) dan metrik kejadian (POD/FAR/CSI), serta walk-forward validation.

## 1. Setup & Helper

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
def mae(a, b):
    return float(np.mean(np.abs(a - b)))

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def r2(a, b):
    return 1 - np.sum((a-b)**2)/np.sum((a-np.mean(a))**2)

def willmott(a, b):
    return 1 - np.sum((a-b)**2)/np.sum((np.abs(b-np.mean(a))+np.abs(a-np.mean(a)))**2)

def metrik_kejadian(y_true, y_pred):
    tp = np.sum((y_pred==1) & (y_true==1))
    fp = np.sum((y_pred==1) & (y_true==0))
    fn = np.sum((y_pred==0) & (y_true==1))
    pod = tp/(tp+fn) if (tp+fn) else np.nan
    far = fp/(tp+fp) if (tp+fp) else np.nan
    csi = tp/(tp+fp+fn) if (tp+fp+fn) else np.nan
    return {"POD": round(pod,3), "FAR": round(far,3), "CSI": round(csi,3)}

print("helpers siap")

## 2. Data Sintetik + Model Overfit

Kita sulit 'memaksa' overfit pada data sederhana; gunakan model relatif besar + banyak epoch tanpa regularisasi pada data ber-noise.

In [ ]:
n = 400
x = np.linspace(-2, 2, n)
y_true = np.sin(3*x)  # pola non-linear
y = y_true + 0.4*np.random.randn(n)

X = x.reshape(-1, 1)
ntr = int(n*0.7); nva = int(n*0.15)
Xtr, ytr = X[:ntr], y[:ntr]
Xva, yva = X[ntr:ntr+nva], y[ntr:ntr+nva]
Xte, yte = X[ntr+nva:], y[ntr+nva:]
print("train", Xtr.shape, "val", Xva.shape, "test", Xte.shape)

In [ ]:
def build_model(reg=False, drop=0.0):
    layers = []
    if reg:
        layers.append(tf.keras.layers.Dense(64, activation="relu", input_shape=(1,),
                                           kernel_regularizer=tf.keras.regularizers.l2(1e-3)))
    else:
        layers.append(tf.keras.layers.Dense(64, activation="relu", input_shape=(1,)))
    if drop > 0:
        layers.append(tf.keras.layers.Dropout(drop))
    layers.append(tf.keras.layers.Dense(64, activation="relu"))
    if drop > 0:
        layers.append(tf.keras.layers.Dropout(drop))
    layers.append(tf.keras.layers.Dense(1))
    m = tf.keras.Sequential(layers)
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return m

m_over = build_model()
h_over = m_over.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=800, batch_size=32, verbose=0)
print("Tanpa regularisasi -> val MAE final:", round(h_over.history["val_mae"][-1], 4))

## 3. Bandingkan Dengan Regularisasi (L2 + Dropout + EarlyStopping)

In [ ]:
cb = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)]
m_reg = build_model(reg=True, drop=0.3)
h_reg = m_reg.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=800, batch_size=32,
                  callbacks=cb, verbose=0)
print("Dengan regularisasi -> val MAE final:", round(h_reg.history["val_mae"][-1], 4))

In [ ]:
plt.figure(figsize=(10, 4))
for i, (h, lbl) in enumerate([(h_over, "tanpa regul"), (h_reg, "dengan regul")]):
    plt.subplot(1, 2, i+1)
    plt.plot(h.history["loss"], label="train")
    plt.plot(h.history["val_loss"], label="val")
    plt.title(lbl); plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.tight_layout()
plt.savefig("learning_curve_compare.png", dpi=150)
plt.show()

## 4. Metrik Regresi

Hitung MAE, RMSE, R², Willmott pada test untuk kedua model.

In [ ]:
for model, lbl in [(m_over,"tanpa regul"),(m_reg,"dengan regul")]:
    p = model.predict(Xte, verbose=0).ravel()
    print(f"{lbl}: MAE={mae(yte,p):.4f} RMSE={rmse(yte,p):.4f} R2={r2(yte,p):.4f} Willmott={willmott(yte,p):.4f}")

## 5. Metrik Kejadian (POD/FAR/CSI) pada Klasifikasi Mini

Contoh: 1000 hari, 40 hari hujan deras. Model memprediksi 30 kali, 10 benar.

In [ ]:
np.random.seed(7)
y_true = np.zeros(1000); y_true[:40] = 1
np.random.shuffle(y_true)
y_pred = np.zeros(1000)
idx = np.random.choice(1000, 30, replace=False)  # 30 prediksi 'deras'
y_pred[idx[:10]] = 1                              # 10 benar, 20 false alarm

print(metrik_kejadian(y_true, y_pred))
print("Interpretasi: POD rendah => banyak kejadian terlewat; FAR tinggi => banyak alarm palsu.")

## 6. Walk-Forward (Contoh Kecil)

Data deret sintetik; kita latih model baru di tiap fold dan hitung MAE rata-rata.

In [ ]:
t = np.arange(0, 500)
ts = np.sin(2*np.pi*t/12.42) + 0.05*np.random.randn(len(t))
Xw = np.column_stack([ts[:-2], ts[1:-1]])
yw = ts[2:]

horizon = 30; wins = 120
folds = []
for start in range(0, len(Xw) - wins - horizon, horizon):
    i_end = start + wins
    m = tf.keras.Sequential([
        tf.keras.layers.Dense(8, activation="relu", input_shape=(2,)),
        tf.keras.layers.Dense(8, activation="relu"),
        tf.keras.layers.Dense(1)])
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    m.fit(Xw[start:i_end], yw[start:i_end], epochs=40, batch_size=32, verbose=0)
    pred = m.predict(Xw[i_end:i_end+horizon], verbose=0).ravel()
    folds.append(mae(yw[i_end:i_end+horizon], pred))

print("Fold MAE:", [round(f,4) for f in folds])
print("Rata-rata MAE walk-forward:", round(float(np.mean(folds)), 4))

## 7. Latihan Mini

1. Kurangi jumlah neuron/depth pada `build_model` — apakah overfit berkurang? Apa trade-off-nya?
2. Naikkan L2 & dropout — apa efeknya pada train vs val MAE? Kapan 'kelewat batas'?
3. Hitung CSI untuk threshold berbeda pada tugas klasifikasi (ambil dari Bab 3).
4. Ganti walk-forward dengan k-fold acak pada data waktu — diskusikan mengapa hasilnya menyesatkan.